# Studying the optical response with **`exciting`** using *RT-TDDFT*
By: Dmitry Tumakov

**<span style="color:firebrick">Note</span>**: Due to time constraints, all results used in this tutorial have been precomputed. Therefore, the corresponding data are retrieved from the [**NOMAD**](https://nomad-lab.eu/prod/v1/gui/search/entries) database. Otherwise, `exciting` would need to be installed and compiled, and the appropriate environment would need to be configured. More information about exciting and its usage can be found on [exciting webpage](https://www.exciting-code.org/).

<hr style="border:2px solid #DDD"> </hr>

## Introduction and physical background
To study the optical response of a system, we employ the real-time approach. Within this framework, we apply an external time-dependent perturbation to a system in equilibrium and use the resulting time-dependent observables to obtain the optical response functions.

In this example, we demonstrate how to calculate the dielectric function, which is proportional to the experimental absorption spectrum. To probe the entire energy range at once, we use an external perturbation in the form of a Dirac delta function, which is constant in the frequency domain.

The coupling of an electronic system to a long-wavelength external field can generally be described using either the length gauge or the velocity gauge. In this work, we aim to compute the low-frequency absorption spectrum, which is known to be difficult to converge when using the velocity gauge. However, the straightforward application of the length-gauge interaction term, $\mathbf{r} \cdot \mathbf{E}(t)$, breaks the periodic boundary conditions (PBC) typically employed for crystalline systems. To address these issues, we adopt the time-dependent effective Schrödinger equation approach, which provides a formulation of the length gauge that is compatible with periodic boundary conditions.

We consider the set of the following equations of motion for the spatially periodic parts of the valence **KS** wavefunctions $v_{j\mathbf{k}}(\mathbf{r},t) = e^{- \mathrm{i} \mathbf{k} \cdot \mathbf{r}} \psi_{j\mathbf{k}}(\mathbf{r},t)$:
\begin{equation}
  \tag{1}
  \mathrm{i} \hspace{0.2mm} \frac{\partial}{\partial t} \, v_{j\mathbf{k}}(\mathbf{r},t)  = \left[h^{\mathbf{k}}_0(\mathbf{r}) + \Delta h^{\mathbf{k}}(\mathbf{r}, t) + \mathrm{i} \mathbf{E}(t) \cdot \tilde{\partial}_{\mathbf{k}} \right] v_{m\mathbf{k}}(\mathbf{r},t),
\end{equation}
where a **KS** state is labeled by an index $j = 1 \dots N_{\rm occupied}$ and associated with a wavevector $\mathbf{k}$. The field-coupling operator $\mathrm{i} \mathbf{E}(t) \cdot \tilde{\partial}_{\mathbf{k}}$ takes form of a gauge-covariant derivative in $\mathbf{k}$ space. The stationary term in the Hamiltonian corresponds to the ground (unperturbed) state of a system:
\begin{equation}
\tag{2}
h^{\mathbf{k}}_0(\mathbf{r}) = e^{- \mathrm{i} \mathbf{k} \cdot \mathbf{r}} \left\{ -\frac{1}{2} \nabla^2 + V_{\scriptscriptstyle \textrm{KS}}(\mathbf{r}, t = 0) \right\} e^{ \mathrm{i} \mathbf{k} \cdot \mathbf{r}}, 
\end{equation}
and the time-dependent part accounts for the changes in the effective **KS** potential:
\begin{equation}
\tag{3}
\Delta h^{\mathbf{k}}(\mathbf{r}, t) = e^{- \mathrm{i} \mathbf{k} \cdot \mathbf{r}} \left\{ V_{\scriptscriptstyle \textrm{KS}}(\mathbf{r}, t) - V_{\scriptscriptstyle \textrm{KS}}(\mathbf{r}, t = 0) \right\} e^{ \mathrm{i} \mathbf{k} \cdot \mathbf{r}}.
\end{equation}
In this example, for didactic reasons, we assume the independent particle approximation (**IPA**):
\begin{equation}
\tag{4}
V_{\scriptscriptstyle \textrm{KS}}(\mathbf{r}, t) = V_{\scriptscriptstyle \textrm{KS}}(\mathbf{r}, t = 0).
\end{equation}

Before the time propagation, we solve the ground-state eigenvalue problem
\begin{equation}
\tag{5}
h^{\mathbf{k}}_0(\mathbf{r}) \, v^0_{n\mathbf{k}}(\mathbf{r}) = \epsilon_{n\mathbf{k}} \, v^0_{n\mathbf{k}}(\mathbf{r}).
\end{equation}

To solve Eq. (1) numerically in **`exciting`**, we expand a **KS** state over a finite basis of 
$N_{\scriptscriptstyle \textrm{KS}} = N_{\rm occupied} + N_{\rm empty}$ periodic parts of the unperturbed **KS** states $v^0_{n\mathbf{k}}(\mathbf{r})$:
\begin{equation}
\tag{6}
v_{j\mathbf{k}}(\mathbf{r},t) = \sum_{n = 1}^{N_{\rm KS}} c_{nj\mathbf{k}}(t) v^0_{n\mathbf{k}}(\mathbf{r}).
\end{equation}
Substituting Eq. (6) into Eq. (1) leads to a set of $N_{\rm occupied}$ differential equations for the expansion coefficients, which are evolved in time using the 4th-order Runge-Kutta method.

During the propagation, macroscopic time-dependent polarization is calculated using the modern theory of polarization as
\begin{equation}
\tag{7}
P_{\alpha}(t) = \frac{ \mathbf{a}_{\alpha}}{\pi \Omega N_{\mathbf{k}_{\alpha}^{\perp}}} \sum_{\mathbf{k}_{\alpha}^{\perp}} \,\mathrm{Im} \hspace{0.2mm} \left[ \ln\prod_{i = 1}^{N_{\mathbf{k}_{\alpha}} - 1} {\rm det} \left \langle v_{m \mathbf{k}} | v_{n \mathbf{k}_{\alpha}^{+}} \right \rangle \right],
\end{equation}
where $\Omega$ is the unit cell volume, $\mathbf{k}_{\alpha}^{+} = \mathbf{k}_{\alpha} + \Delta \mathbf{k}_\alpha$, and $N_{\mathbf{k}_{\alpha}}$ and $N_{\mathbf{k}_{\alpha}^{\perp}}$ denote the number of $\mathbf{k}$ points along cell direction $\alpha$ and orthogonal to it, respectively.


Using the macroscopic polarization, we obtain the dielectric function components as
\begin{equation}
\tag{8}
\varepsilon_{ij}(\omega) = \delta_{ij} + 4 \pi P_i(\omega) / E_j(\omega).
\end{equation}


## Setup and usage of exciting

#### Preliminary GS calculations

As a preliminary step to calculate properties using the RT approach, a ground-state calculation has to be performed. In this tutorial we consider as an example **$\beta$-Ga<sub>2</sub>O<sub>3</sub>**.

An `exciting` input file needs to contain the <code><span style="color:green">structure</span></code> element in which we include the lattice parameter, basis vectors, and atomic positions and the <code><span style="color:green">groundstate</span></code> element where we include parameters responsible for **DFT** ground-state calculations. We also need so called species files in which we define the augmented plane-wave and local-orbital basis. Have in mind that the basis within species files needs to be converged.

We can read the ground-state input from file using the exciting python interface **excitingtools**. Details on this can be found in the [**excitingtools tutorial**](https://www.exciting-code.org/uploads/exciting/tutorial_notebooks/tutorial_excitingtools.html).

In [ ]:
from excitingtools import ExcitingInputXML

exciting_input = ExcitingInputXML.from_xml('data/gs_input.xml')

How such an ground-state calculation can be run, you can find out in the ground-state tutorial. 
The main output of the **DFT** calculation for subsequent excited states calculations are stored in **STATE.OUT** and **EFERMI.OUT**. 

Since we use the mock runner and the binary file **STATE.OUT** is not transferable between maschines and compilers, we dont fetch **DFT** results seperately and directly proceed with the **RT-TDDFT** calculations. But we assume, we have the neccesary ground state calculation done, therefore we can skip the recomputation of the ground state by seting the following parameter in the input file. 

In [ ]:
exciting_input.groundstate.do = "skip"

#### RT-TDDFT input

To perform the RT-TDDFT calculation, we add <code><span style="color:green">xs</span></code> element to the input file, and set <code><span style="color:mediumblue">xstype</span>=<span style="color:firebrick">"RT-TDDFT"</span></code>.

In [ ]:
from excitingtools import ExcitingXSInput

xs = {'xstype': 'RT-TDDFT',
      'ngridk': [8, 8, 8],
      'rgkmax': 5.0,
      'nosym': True,
      'reducek': False,
      'nempty': 12,
      'realTimeTDDFT': {'propagator': 'RK4', 'timeStep': 0.25, 'endTime': 4000.0,
                        'basis': 'unperturbedKS', 'fieldCoupling': 'berryPhase', 'eeInteraction': 'IPA', 
                        'numberOfFrozenStates': 16,
                        'laser': {'kick': [{'amplitude': 0.0001, 't0': 0.5, 'width': 0.15, 'direction': 'x'}]}}}

exciting_input.xs = ExcitingXSInput(**xs)


**<span style="color:firebrick">Please note</span>**: numerical input parameters such as <code><span style="color:mediumblue">ngridk</span></code>, <code><span style="color:mediumblue">rgkmax</span></code> and <code><span style="color:mediumblue">nempty</span></code> specified for the **RT-TDDFT** calculation can differ from their **groundstate** counterparts, while their meaning stays the same.

The symmetry of the system "crystal + external field" is lower than the symmetry of the crystal alone. Since partial symmetry reduction is not yet implemented in **`exciting`**, we set <code><span style="color:mediumblue">nosym</span>=<span style="color:firebrick">"True"</span></code> and <code><span style="color:mediumblue">reducek</span>=<span style="color:firebrick">"False"</span></code> to treat each $\mathbf{k}$ point as a unique one.

The setup of a RT-TDDFT run is prepared via the <code><span style="color:green">realTimeTDDFT</span></code> element. <code><span style="color:mediumblue">propagator</span></code>, <code><span style="color:mediumblue">timeStep</span></code> and <code><span style="color:mediumblue">endTime</span></code> parameters say we employ the Runge-Kutta-4 propagator with a time step of 0.25 a.u., and stop the propagation at time $t$ = 4000 a.u. ($\approx$ 97 fs). <code><span style="color:mediumblue">basis</span></code>=<span style="color:firebrick">unperturbedKS</span></code> forces the code to use the ground-state **KS** wavefunctions to discretize the time-dependent problem (see Eq. (6)), and not the **LAPW+lo** basis set. In this case, basis size is governed by the value of <code><span style="color:mediumblue">nempty</span></code> parameter, which is set to 12 in this example. <code><span style="color:mediumblue">fieldCoupling</span></code>=<span style="color:firebrick">berryPhase</span></code> sets the coupling with the external perturbation to be in the PBC-compatible length gauge form $\mathrm{i} \mathbf{E}(t) \cdot \tilde{\partial}_{\mathbf{k}}$, and <code><span style="color:mediumblue">eeInteraction</span></code>=<span style="color:firebrick">IPA</span></code> forces the **IPA** (see Eq. (4)). 

To speed up the calculations, we reduce the number of states, which will be evolved in time, by setting <code><span style="color:mediumblue">numberOfFrozenStates</span></code>=<span style="color:firebrick">16</span></code>, so the index $j$ in Eqs. (1) runs from 17 to $N_{\rm occupied}$ = 60 for the considered system. We choose this value from examining the **EIGVAL.OUT** file we got in the GS step: we are interested in the low-frequency (up to around 20 eV) linear response of the system, and 16 lowest-lying valence states are separated by more than 80 eV from the conduction band, which makes them irrelevant for the current example.

The element <code><span style="color:mediumblue">laser</span></code> defines the parameters for a delta-like kick with a small finite broadening:
\begin{equation}
\mathbf{E}(t) = \mathbf{E}_0 \frac{15}{16}\left( \frac{t-t_0}{w} +1 \right)^2 \left( \frac{t-t_0}{w} -1 \right)^2.
\end{equation}

## Execution of the code

Now, we build a directory for the calculation

In [ ]:
from pathlib import Path
rt_tddft_dir = Path("rt_tddft_tutorial")
rt_tddft_dir.mkdir(exist_ok=True)

and safe the generate input into an xml file. 

In [ ]:
input_xml = rt_tddft_dir / "input.xml"
exciting_input.write(input_xml)

With that, we can run exciting using **excitingscripts**. This python library contains a lot of helpful scripts for running exciting and postprocessing exciting outputs. In this tutorial, we use the mock runner to fetch the pre-computed data from [**NOMAD**](https://nomad-lab.eu/prod/v1/gui/search/entries).

In [ ]:
%%bash
cd rt_tddft_tutorial
python -m excitingscripts.dpg2026.mock_execute_single TDDFT --overwrite
cd ..

## Parsing and visualization of results
In this example, the one-step calculation preceding the evolution of the **KS** states generates output files ending with **_RTTDDFT.OUT**. They contain the same information as those produced during a standard self-consistent cycle triggered by the <code><span style="color:green">groundstate</span></code> element. The exception is **POLARIZATION_RTTDDFT.OUT**, which contains the system’s time-dependent macroscopic polarization.

The main result of interest in this example is the macroscopic polarization, stored in **POLARIZATION_RTTDDFT.OUT**. You can visualize its $x$ component by running the following command:

In [ ]:
%%bash
cd rt_tddft_tutorial
python -m excitingscripts.plot.multitask --pol -f POLARIZATION_RTTDDFT.OUT --x --xlim 20 4000 --ylim -1.5e-7 1.5e-7  --plot_name POLARIZATION
cd ..

In [ ]:
from IPython.display import Image

Image(filename='rt_tddft_tutorial/POLARIZATION.png') 

The resulting figure is saved as **POLARIZATION.png** in the current directory.

Now, we obtain the $xx$ component of the dielectric function. Note that we use a Lorentzian broadening of 0.2 eV on this step to emulate the real experimental conditions.

In [ ]:
%%bash
cd rt_tddft_tutorial
python -m excitingscripts.plot.multitask --preprocess --get_eps_from_polarization --xx -f ELECTRIC_FIELD.OUT -f POLARIZATION_RTTDDFT.OUT --wcut 0.007 -o EPSILON.OUT
python -m excitingscripts.plot.multitask --imag_eps -f EPSILON.OUT  --xlim 0 22 --ylim 0 6 --plot_name IMAGEPS_XX
cd ..

In [ ]:
Image(filename='rt_tddft_tutorial/IMAGEPS_XX.png')


<hr style="border:2px solid #DDD"> </hr>
This concludes our tutorial for the computation of optical response with the rt-TDDFT method. You are encouraged to checkout the other methods implemented in exciting via the tutorials in this suite.
<hr style="border:2px solid #DDD"> </hr>